**03 - Gold and Data Warehouse (star schema)**

Builds three dimensions and one fact table from the Silver file, saves them as CSV files in the `gold` folder, then loads them into a SQLite database (`airbnb_dwh.db`) with primary keys, foreign keys and a city summary view.

In [ ]:
import os
import sqlite3
import pandas as pd

SILVER_CSV = "airbnb_silver.csv"
GOLD_DIR = "gold"
DB_PATH = "airbnb_dwh.db"
SCRAPE_DATE = "2026-10-19"

os.makedirs(GOLD_DIR, exist_ok=True)

silver = pd.read_csv(SILVER_CSV, dtype={"listing_id": "string"})
silver["reviews_count"] = silver["reviews_count"].astype("Int64")
print(silver.shape)

In [ ]:
dim_city = silver[["city", "country"]].drop_duplicates().sort_values("city").reset_index(drop=True)
dim_city.insert(0, "city_key", dim_city.index + 1)

dim_city.to_csv(os.path.join(GOLD_DIR, "dim_city.csv"), index=False, encoding="utf-8-sig")
dim_city

In [ ]:
scrape_date = pd.Timestamp(SCRAPE_DATE)

dim_date = pd.DataFrame({
    "date_key": [int(scrape_date.strftime("%Y%m%d"))],
    "full_date": [scrape_date.date()],
    "year": [scrape_date.year],
    "month": [scrape_date.month],
    "month_name": [scrape_date.strftime("%B")],
    "day": [scrape_date.day],
})

dim_date.to_csv(os.path.join(GOLD_DIR, "dim_date.csv"), index=False, encoding="utf-8-sig")
dim_date

In [ ]:
dim_listing = silver[["listing_id", "name", "url", "bedrooms", "location", "is_superhost", "guest_favorite"]]
dim_listing = dim_listing.drop_duplicates(subset="listing_id").reset_index(drop=True)
dim_listing.insert(0, "listing_key", dim_listing.index + 1)

dim_listing.to_csv(os.path.join(GOLD_DIR, "dim_listing.csv"), index=False, encoding="utf-8-sig")

print("rows:", len(dim_listing))
print("duplicate listing_id:", int(dim_listing["listing_id"].duplicated().sum()))
dim_listing.head()

In [ ]:
fact = silver.merge(dim_city, on=["city", "country"]).merge(dim_listing[["listing_key", "listing_id"]], on="listing_id")
fact["date_key"] = dim_date.loc[0, "date_key"]

fact_listing_snapshot = fact[[
    "listing_key", "city_key", "date_key",
    "price_per_night_usd", "rating", "reviews_count", "has_reviews",
]].sort_values("listing_key").reset_index(drop=True)

fact_listing_snapshot.to_csv(os.path.join(GOLD_DIR, "fact_listing_snapshot.csv"), index=False, encoding="utf-8-sig")

print("rows:", len(fact_listing_snapshot))
print("null keys:", int(fact_listing_snapshot[["listing_key", "city_key", "date_key"]].isna().sum().sum()))
print("duplicate listing_key:", int(fact_listing_snapshot["listing_key"].duplicated().sum()))
print("orphan city_key:", int((~fact_listing_snapshot["city_key"].isin(dim_city["city_key"])).sum()))
print("orphan listing_key:", int((~fact_listing_snapshot["listing_key"].isin(dim_listing["listing_key"])).sum()))

fact_listing_snapshot.head()

In [ ]:
city_summary = (
    fact_listing_snapshot
    .merge(dim_city, on="city_key")
    .merge(dim_listing[["listing_key", "is_superhost", "guest_favorite"]], on="listing_key")
    .groupby(["city_key", "city", "country"], as_index=False)
    .agg(
        listings=("listing_key", "count"),
        avg_price=("price_per_night_usd", "mean"),
        median_price=("price_per_night_usd", "median"),
        avg_rating=("rating", "mean"),
        superhost_pct=("is_superhost", "mean"),
        guest_favorite_pct=("guest_favorite", "mean"),
        new_listings=("has_reviews", lambda s: (~s).sum()),
    )
)

city_summary["superhost_pct"] = (city_summary["superhost_pct"] * 100).round(1)
city_summary["guest_favorite_pct"] = (city_summary["guest_favorite_pct"] * 100).round(1)
city_summary[["avg_price", "median_price", "avg_rating"]] = city_summary[["avg_price", "median_price", "avg_rating"]].round(2)

city_summary.to_csv(os.path.join(GOLD_DIR, "city_summary.csv"), index=False, encoding="utf-8-sig")

city_summary.sort_values("avg_price", ascending=False)

In [ ]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
CREATE TABLE dim_city (
    city_key INTEGER PRIMARY KEY,
    city TEXT NOT NULL,
    country TEXT NOT NULL
);

CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date TEXT NOT NULL,
    year INTEGER NOT NULL,
    month INTEGER NOT NULL,
    month_name TEXT NOT NULL,
    day INTEGER NOT NULL
);

CREATE TABLE dim_listing (
    listing_key INTEGER PRIMARY KEY,
    listing_id TEXT NOT NULL UNIQUE,
    name TEXT,
    url TEXT,
    bedrooms INTEGER,
    location TEXT,
    is_superhost INTEGER,
    guest_favorite INTEGER
);

CREATE TABLE fact_listing_snapshot (
    listing_key INTEGER NOT NULL REFERENCES dim_listing(listing_key),
    city_key INTEGER NOT NULL REFERENCES dim_city(city_key),
    date_key INTEGER NOT NULL REFERENCES dim_date(date_key),
    price_per_night_usd REAL,
    rating REAL,
    reviews_count INTEGER,
    has_reviews INTEGER,
    PRIMARY KEY (listing_key, date_key)
);

CREATE VIEW vw_city_summary AS
SELECT
    c.city_key,
    c.city,
    c.country,
    COUNT(*) AS listings,
    ROUND(AVG(f.price_per_night_usd), 2) AS avg_price,
    ROUND(AVG(f.rating), 2) AS avg_rating,
    ROUND(100.0 * AVG(l.is_superhost), 1) AS superhost_pct,
    ROUND(100.0 * AVG(l.guest_favorite), 1) AS guest_favorite_pct,
    SUM(CASE WHEN f.has_reviews = 0 THEN 1 ELSE 0 END) AS new_listings
FROM fact_listing_snapshot f
JOIN dim_city c ON c.city_key = f.city_key
JOIN dim_listing l ON l.listing_key = f.listing_key
GROUP BY c.city_key, c.city, c.country;
""")

pd.read_csv(os.path.join(GOLD_DIR, "dim_city.csv")).to_sql("dim_city", conn, if_exists="append", index=False)
pd.read_csv(os.path.join(GOLD_DIR, "dim_date.csv")).to_sql("dim_date", conn, if_exists="append", index=False)
pd.read_csv(os.path.join(GOLD_DIR, "dim_listing.csv"), dtype={"listing_id": "string"}).to_sql("dim_listing", conn, if_exists="append", index=False)
pd.read_csv(os.path.join(GOLD_DIR, "fact_listing_snapshot.csv")).to_sql("fact_listing_snapshot", conn, if_exists="append", index=False)
conn.commit()

for t in ["dim_city", "dim_date", "dim_listing", "fact_listing_snapshot"]:
    print(t, conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])

print("foreign key problems:", len(conn.execute("PRAGMA foreign_key_check").fetchall()))

pd.read_sql("SELECT * FROM vw_city_summary ORDER BY avg_price DESC", conn)

In [ ]:
conn.close()
print("DWH saved to", DB_PATH)